# KEEPCODING - AI Engineering - M12 Prompt Engineering

## Clase 2 - Lab 3 - Reasoning Prompts en 2026

En 2026 el razonamiento es un **dial**, no una clase de modelo aparte. Las familias frontera (GPT-5, Claude 4.7, Gemini 3, DeepSeek) llevan el razonamiento integrado y tu decides cuanto gastan con `reasoning_effort`:

- **Effort bajo (`minimal` / `low`)**: el modelo responde casi directo. Rapido y barato. Es el "modo chat de toda la vida": si quieres que razone, le guias tu (Chain of Thought manual, few-shot).
- **Effort alto (`medium` / `high`)**: el modelo genera **tokens de razonamiento internos** (no te llegan) antes de contestar. Mas caro y lento, pero resuelve matematicas, logica y codigo que el modo directo falla.
- **Reasoners dedicados** (`o4-mini`, `o3`, `DeepSeek-R1`): familias optimizadas para razonar siempre; el dial va de `low` a `high`, no tienen modo "directo".

El gran cambio: **los prompts que funcionan a effort bajo son a menudo contraproducentes a effort alto**, y al reves. En este lab comparamos los dos extremos con un puzzle logico: `gpt-5-mini` a `reasoning_effort='minimal'` (rapido) vs `o4-mini` razonando.

### Que vas a aprender

- Que cambia "por dentro" al subir `reasoning_effort` y por que.
- 5 patrones do/don't para prompts cuando el modelo razona.
- Comparacion practica: mismo puzzle a effort minimal vs un reasoner dedicado.
- Cost/latency trade-off y cuando merece la pena subir el dial.
- Matriz de decision: effort bajo vs alto, tarea por tarea.

> Prerequisito: cuenta en OpenAI con acceso a `o4-mini`. Si tu API key no lo tiene, el codigo te avisara y puedes saltar las celdas que lo usan (o sustituirlo por otro reasoning model compatible con la API OpenAI: `deepseek-reasoner` via Together, por ejemplo).

> **Cómo seguir este lab**
>
> Este lab compara modelos chat (gpt-5-mini) vs modelos reasoning (o4-mini) en un puzzle lógico real. Es el más corto y el más concentrado del módulo: ~26 celdas.
>
> Tres usos posibles:
>
> 1. **Lectura rápida** · ve la matriz de decisión y los patrones do/don't. ~10 minutos.
> 2. **Ejecución sin tocar** · pega tu API key, ejecuta las 4 comparativas (chat sin CoT, chat con CoT, reasoning sin CoT, reasoning con CoT anti-patrón). Costo total ~$0.05. ~15 minutos.
> 3. **Tu propio puzzle** · sustituye el de Caballeros y Bribones por uno tuyo. Mide cómo cambia la respuesta entre chat y reasoning.
>
> Si tu cuenta no tiene acceso a o4-mini, el notebook detecta el error y sugiere DeepSeek-R1 vía Together como alternativa.

## 1. Setup

### 1.1. Instalar dependencias

In [ ]:
%pip install -qU "openai>=1.50.0"

### 1.2. Cargar la API key de OpenAI

Como en el resto de labs del curso, intentamos tres fuentes en orden:

1. Variable de entorno `OPENAI_API_KEY` (si ya esta configurada).
2. Secretos de Google Colab.
3. Entrada manual con `getpass` (no muestra la clave en pantalla).

In [ ]:
import os
from getpass import getpass


def cargar_api_key() -> None:
    """Asegura que OPENAI_API_KEY este disponible en el entorno."""
    if os.environ.get("OPENAI_API_KEY"):
        print("OPENAI_API_KEY encontrada en las variables de entorno.")
        return

    try:
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        print("OPENAI_API_KEY cargada desde los secretos de Colab.")
        return
    except Exception:
        pass

    os.environ["OPENAI_API_KEY"] = getpass("Introduce tu OPENAI_API_KEY: ")
    print("OPENAI_API_KEY cargada manualmente.")


cargar_api_key()


### 1.3. Cliente y helpers

Definimos dos helpers, los dos extremos del dial:

- `chat_completion(...)` = el modo **rapido**: `gpt-5-mini` a `reasoning_effort='minimal'`, responde casi directo. (Para modelos no-reasoning como `gpt-4.1` usa `temperature` normal.)
- `reasoning_completion(...)` = el modo que **razona**: usa `reasoning_effort` (`low`/`medium`/`high`) y `max_completion_tokens` en vez de `temperature`.

Tambien medimos **tiempo** y **tokens** para que veas el trade-off real en cada llamada.

In [ ]:
import time
from openai import OpenAI

client = OpenAI()

CHAT_MODEL      = "gpt-5-mini"   # modo rapido: a reasoning_effort='minimal' responde directo
REASONING_MODEL = "o4-mini"      # razona internamente, mas caro y mas lento


def chat_completion(prompt: str, model: str = CHAT_MODEL, temperature: float = 0.0):
    """Modo rapido del dial. Devuelve (texto, tiempo_s, tokens).

    - gpt-5*    -> reasoning_effort='minimal' (responde casi directo, barato).
    - o1/o3/o4  -> reasoner dedicado: ni temperature ni minimal.
    - resto (gpt-4.1, ...) -> chat clasico con temperature.
    """
    t0 = time.time()
    kwargs = {"model": model, "messages": [{"role": "user", "content": prompt}]}
    if model.startswith("gpt-5"):
        kwargs["reasoning_effort"] = "minimal"
    elif model.startswith(("o1", "o3", "o4")):
        pass
    else:
        kwargs["temperature"] = temperature
    resp = client.chat.completions.create(**kwargs)
    dt = time.time() - t0
    return resp.choices[0].message.content, dt, resp.usage


def reasoning_completion(prompt: str, model: str = REASONING_MODEL, effort: str = "medium"):
    """Llama a un reasoning model.

    Notas sobre la API:
    - No acepta temperature (el modelo decide su propio sampling interno).
    - Usa max_completion_tokens, no max_tokens.
    - reasoning_effort: "low" | "medium" | "high". Mas effort = mas tokens internos = mejor calidad y mas coste.
    """
    t0 = time.time()
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        reasoning_effort=effort,
    )
    dt = time.time() - t0
    return resp.choices[0].message.content, dt, resp.usage


## 2. Qué es un reasoning model · intuición

Un chat model tradicional responde token a token directamente. Un **reasoning model** tiene una fase previa: genera **tokens de razonamiento internos** (no te llegan) y solo despues produce la respuesta visible.

```
Chat model      :  prompt -> [token1 token2 token3 ... ] -> respuesta
Reasoning model :  prompt -> [reasoning_tokens_internos] -> [respuesta]
                              \____________________/
                               no los ves, pero los pagas
```

Las consecuencias practicas son grandes:

1. **No le pidas Chain of Thought**. Ya lo hace por dentro. Si le dices "piensa paso a paso", desperdicias tokens y a veces empeora.
2. **No le pongas restricciones de longitud agresivas**. Si limitas `max_completion_tokens` muy bajo, el modelo se queda sin tokens en mitad de su razonamiento interno y devuelve respuesta truncada o vacia.
3. **Dale contexto completo, no recetas paso a paso**. El reasoning model planifica solo. Si le metes una "receta" detallada, le impones un mal plan.
4. **Latencia y coste suben 10-100x**. No lo uses para chats triviales.
5. **Structured output con cuidado**. Hay reasoning models que ya soportan structured outputs (`response_format` JSON schema) pero, segun el modelo, hay que verificarlo. Si no, mejor pedir JSON con un schema en el prompt.

## 3. Comparación práctica · un puzzle lógico

Vamos a usar el mismo problema con los dos modelos. Es un puzzle del estilo "knights and knaves" (caballeros que dicen siempre la verdad y bribones que mienten siempre), no trivial pero corto.

In [ ]:
PUZZLE = """En una isla viven dos tipos de personas:
- Los caballeros, que siempre dicen la verdad.
- Los bribones, que siempre mienten.

Nos encontramos con tres habitantes: A, B y C.

A dice: "B es un bribon."
B dice: "A y C son del mismo tipo."
C dice: "A es un caballero."

Determina que es cada uno (caballero o bribon) y justifica brevemente.
Devuelve solo el resultado en el formato:
A: <caballero|bribon>
B: <caballero|bribon>
C: <caballero|bribon>
"""

print(PUZZLE)


### 3.1. Chat model **sin** Chain of Thought

Le pedimos directamente la respuesta. Es esperable que falle o se equivoque en un puzzle de logica con varios casos.

In [ ]:
respuesta, tiempo, uso = chat_completion(PUZZLE)
print("=== Chat (sin CoT) ===")
print(respuesta)
print(f"\nTiempo: {tiempo:.2f}s")
print(f"Tokens: prompt={uso.prompt_tokens}, completion={uso.completion_tokens}, total={uso.total_tokens}")


### 3.2. Chat model **con** Chain of Thought manual

Mismo prompt + "piensa paso a paso, considera los dos casos posibles para A". Tipico patron 2024 que mejora chat models.

In [ ]:
PROMPT_COT = PUZZLE + """

Piensa paso a paso. Considera primero el caso "A es caballero" y deduce que pasa con B y C. Luego considera el caso "A es bribon" y deduce el otro escenario. Quedate con el unico que es internamente consistente.
"""

respuesta, tiempo, uso = chat_completion(PROMPT_COT)
print("=== Chat + Chain of Thought manual ===")
print(respuesta)
print(f"\nTiempo: {tiempo:.2f}s")
print(f"Tokens: prompt={uso.prompt_tokens}, completion={uso.completion_tokens}, total={uso.total_tokens}")


### 3.3. Reasoning model · SIN CoT manual

Aqui no le decimos "piensa paso a paso". Le mandamos el puzzle pelado. El modelo razonara por dentro.

In [ ]:
try:
    respuesta, tiempo, uso = reasoning_completion(PUZZLE, effort="medium")
    print("=== Reasoning (medium effort, sin CoT manual) ===")
    print(respuesta)
    print(f"\nTiempo: {tiempo:.2f}s")
    print(f"Tokens: prompt={uso.prompt_tokens}, completion={uso.completion_tokens}, total={uso.total_tokens}")
    if hasattr(uso, "completion_tokens_details") and uso.completion_tokens_details is not None:
        print(f"  - de los cuales reasoning_tokens (internos): {getattr(uso.completion_tokens_details, 'reasoning_tokens', 'n/a')}")
except Exception as e:
    print("Tu API key no tiene acceso al reasoning model, o el nombre del modelo ha cambiado:")
    print(repr(e))
    print("\nPuedes sustituir REASONING_MODEL por otro reasoning model accesible (p.ej. via Together: 'deepseek-ai/DeepSeek-R1').")


### 3.4. Reasoning model · **anti-patrón**: añadirle CoT manual

Para mostrar que **el CoT manual sobra**, mandamos el mismo prompt con instrucciones detalladas paso a paso, igual que con el chat. Compara tokens y tiempo.

In [ ]:
try:
    respuesta, tiempo, uso = reasoning_completion(PROMPT_COT, effort="medium")
    print("=== Reasoning + CoT manual (anti-patron) ===")
    print(respuesta)
    print(f"\nTiempo: {tiempo:.2f}s")
    print(f"Tokens: prompt={uso.prompt_tokens}, completion={uso.completion_tokens}, total={uso.total_tokens}")
    if hasattr(uso, "completion_tokens_details") and uso.completion_tokens_details is not None:
        print(f"  - reasoning_tokens (internos): {getattr(uso.completion_tokens_details, 'reasoning_tokens', 'n/a')}")
    print("\nObservacion tipica: tokens totales suben, calidad se mantiene o baja, y has pagado de mas.")
except Exception as e:
    print("Salto la celda (reasoning model no accesible):", repr(e))


### 3.5. Reasoning effort · low vs high

`reasoning_effort` controla cuanto razonamiento interno gasta el modelo. Es el dial mas importante para el coste/calidad.

In [ ]:
for effort in ["low", "medium", "high"]:
    try:
        respuesta, tiempo, uso = reasoning_completion(PUZZLE, effort=effort)
        reasoning_t = "n/a"
        if hasattr(uso, "completion_tokens_details") and uso.completion_tokens_details is not None:
            reasoning_t = getattr(uso.completion_tokens_details, "reasoning_tokens", "n/a")
        print(f"--- effort={effort} ---")
        print(f"Tiempo: {tiempo:.2f}s | completion_tokens={uso.completion_tokens} | reasoning_tokens={reasoning_t}")
        print(respuesta.strip()[:300])
        print()
    except Exception as e:
        print(f"effort={effort}: error -> {e}")


## 4. Patrones do / don't para reasoning models

Resumen de buenas y malas practicas, basado en docs de OpenAI/Anthropic/DeepSeek y experiencia en produccion 2025-2026.

### DO

- **Contexto completo, no receta**. Cuanta mas informacion del problema, mejor planifica el modelo.
- **Plantea el objetivo final, no los pasos**. "Demuestra el teorema X" es mejor que "primero, factoriza..., luego, deriva...".
- **Deja `reasoning_effort` en `medium` por defecto**. Solo sube a `high` si el problema es realmente dificil (matematicas competicion, programacion compleja).
- **Da ejemplos del *output*, no del *razonamiento***. Few-shot del formato final esta bien. Few-shot del proceso de pensar suele estorbar.
- **Estructura el prompt con secciones claras** (`# Contexto`, `# Restricciones`, `# Salida esperada`). El modelo lo aprovecha mejor que un parrafo largo.

### DON'T

- **No le digas "piensa paso a paso"**. Ya lo hace; estas a?adiendo ruido.
- **No le pongas `max_completion_tokens` agresivamente bajo**. El modelo necesita margen para sus tokens internos. Pon al menos 2x lo que esperas de salida visible.
- **No mezcles role-play emocional ("eres un experto que ama...") con razonamiento**. Para chat models ayuda. Para reasoning models suele meter sesgo y reducir calidad.
- **No uses reasoning models para tareas creativas o de tono**. Para escribir copy, mails, posts, etc., un chat model suele dar resultados igual de buenos a una fraccion del coste y la latencia.
- **No le pidas que justifique la respuesta dentro de la salida visible si no la necesitas**. La justificacion ya esta en los reasoning_tokens internos (que no ves). Pedirla duplica trabajo y aumenta latencia.

## 5. Cost / latency trade-off

Numeros tipicos en 2026 (orden de magnitud, mira la pagina de pricing oficial para los exactos):

| Aspecto | gpt-5-mini · minimal | o4-mini · medium/high |
|---|---|---|
| Latencia por respuesta | 1-3 s | 10-60 s |
| Coste por 1M output tokens | ~0.6 USD | ~4-8 USD |
| Tokens facturados por respuesta | solo los visibles | visibles + reasoning_tokens internos |
| Calidad en aritmetica / logica | media | alta |
| Calidad en escritura creativa | alta | media |

Implicacion practica: **subir el effort puede costar 10-100x mas por consulta**. No es un sustituto general del modo rapido, es una herramienta para subconjuntos concretos de problemas.

### Heuristica simple

Si tu tarea cae en uno de estos cubos, **sube el effort** (o un reasoner dedicado):

- Demostraciones matematicas o problemas de competicion.
- Planificacion multi-paso (agentes de tareas largas, robotica).
- Codigo con razonamiento global (refactors grandes, bug hunting).
- Decisiones criticas donde un error cuesta mucho dinero (compliance, medico, legal).

Si tu tarea es algo de esto, **deja el effort bajo** (modo rapido):

- Chatbots conversacionales.
- Resumen, traduccion, reescritura.
- Generacion de copy, emails, posts.
- Extraccion estructurada simple (NER, JSON desde texto corto).
- Cualquier cosa donde la latencia importe (<3s).

## 6. Structured output con reasoning models

A finales de 2025 los reasoning models de OpenAI (o4, o4-mini) ya soportan `response_format` con JSON Schema, igual que los chat models. Pero hay matices:

- El esquema cuenta dentro de los tokens internos: si tu schema es enorme, gastas mas reasoning_tokens.
- Algunos modelos viejos (o1-preview, DeepSeek-R1 release inicial) **no** soportan `response_format`. Hay que pedir JSON via prompt y parsear.

Ejemplo seguro: pedimos un schema simple. Si falla, hacemos fallback a parsing de texto.

In [ ]:
import json

JSON_PROMPT = PUZZLE + """

Devuelve el resultado en JSON estricto con esta forma:
{"A": "caballero" o "bribon", "B": "caballero" o "bribon", "C": "caballero" o "bribon"}
"""

def reasoning_json(prompt: str, model: str = REASONING_MODEL):
    """Intenta structured output con reasoning model. Fallback a parseo de texto."""
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            reasoning_effort="medium",
            response_format={"type": "json_object"},
        )
        return json.loads(resp.choices[0].message.content), "structured"
    except Exception as e:
        print(f"Structured output no disponible ({e}). Fallback a parseo.")
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            reasoning_effort="medium",
        )
        text = resp.choices[0].message.content
        # parse muy basico: busca el primer { ... } en la respuesta
        start = text.find("{")
        end   = text.rfind("}")
        if start != -1 and end != -1:
            return json.loads(text[start:end+1]), "fallback"
        return text, "raw"


try:
    out, mode = reasoning_json(JSON_PROMPT)
    print(f"Modo: {mode}")
    print(json.dumps(out, indent=2, ensure_ascii=False))
except Exception as e:
    print("Reasoning model no disponible:", e)


## 7. Matriz de decisión · chat vs reasoning

| Caso de uso | Recomendacion | Por que |
|---|---|---|
| Chatbot de atencion al cliente | **Chat** | Latencia critica, problemas tipicos no requieren razonamiento profundo. |
| Resumen de documentos largos | **Chat** | Tarea esencialmente compresiva, no de razonamiento. |
| Generacion de copy / email / post | **Chat** | El reasoning model tiende a escribir mas plano. |
| Extraccion estructurada (JSON desde texto corto) | **Chat** | Suficiente, mas rapido y barato. |
| Clasificacion / sentiment | **Chat** | Casi siempre suficiente con un buen prompt o few-shot. |
| Demostracion matematica / problema de competicion | **Reasoning** | La calidad sube de forma muy clara. |
| Planificacion multi-paso (agente, scheduling) | **Reasoning** | Mejor descomposicion implicita del problema. |
| Debug complejo / refactor de codigo grande | **Reasoning** | Razonamiento global sobre dependencias. |
| Decisiones criticas (legal, medico, compliance) | **Reasoning** | El coste del error supera al coste del modelo. |
| Generacion de tests unitarios | **Chat** suele bastar | Casos comunes; usa reasoning solo si los tests cubren casos adversarios. |
| Tarea con latencia <1s requerida | **Chat** obligatorio | Reasoning no llega a esa latencia. |

> Patron habil en 2026: **router de modelos**. Un chat model barato decide si una pregunta es "razonamiento-pesada" y, si lo es, la envia al reasoning model. Si no, la responde directamente. Ahorra mucho coste con caida minima de calidad.

## 8. Recap

- El razonamiento es un dial (`reasoning_effort`): el mismo modelo responde directo (rapido/barato) o piensa (caro/lento). No es "mejor", es otra herramienta para otros problemas.
- Las dos reglas mas importantes son: **no les des CoT manual** y **dales contexto completo**.
- Cuestan y tardan 10-100x mas: usalos solo donde la calidad extra compensa.
- Para 95% de los casos, effort bajo + un buen prompt sigue siendo lo correcto.

### Retos para practicar

1. Pasa al menos 3 prompts tuyos de produccion por la matriz de decision. Decide effort bajo o alto para cada uno.
2. Coge un problema que actualmente resuelves con chat + CoT muy elaborado. Mandalo a un reasoning model **pelado** y compara calidad / coste / latencia.
3. Construye un mini-router: una funcion que mira la pregunta y decide el modelo. Usa el modo rapido (effort minimal) como "clasificador de complejidad".